# 00 · Exploração da base SIAC 156

Perfil de três snapshots diários consecutivos (18, 19 e 20/10/2026) do conjunto
[SIAC 156](https://dadosabertos.curitiba.pr.gov.br/conjuntodado/detalhe/?chave=0d5a7b06-3940-4be9-876e-bc8f23e96530),
baixados do espelho C3SL. Objetivos da fase 1:

1. Confirmar encoding, separador e formato das datas.
2. Perfilar nulos, cardinalidade e domínios.
3. Verificar se existe uma combinação de colunas que identifique a solicitação.
4. Entender como um snapshot difere do anterior.

Os arquivos ficam em `data/raw/` (fora do git). Para reproduzir:

```bash
B=http://dadosabertos.c3sl.ufpr.br/curitiba/156
for d in 18 19 20; do curl -o data/raw/2026-10-${d}_156_-_Base_de_Dados.csv $B/2026-10-${d}_156_-_Base_de_Dados.csv; done
```

In [1]:
from pathlib import Path
import duckdb

RAW = Path("../data/raw")
DIAS = ["2026-10-18", "2026-10-19", "2026-10-20"]
COLS = "Tipo,Orgao,DataCriacao,Assunto,Subdivisao,Situacao,Logradouro,Bairro,Regional,DataResposta,Origem"
ESTAVEIS = "Tipo,Orgao,DataCriacao,Assunto,Subdivisao,Logradouro,Bairro,Regional,Origem"

con = duckdb.connect()
q = lambda sql: con.sql(sql).df()

## 1. Formato do arquivo

In [2]:
for d in DIAS:
    f = RAW / f"{d}_156_-_Base_de_Dados.csv"
    head = f.read_bytes()[:400]
    print(f.name, f"{f.stat().st_size / 1e6:.1f} MB", "BOM UTF-8:", head.startswith(b"\xef\xbb\xbf"), "CRLF:", b"\r\n" in head)
print(head.decode("utf-8-sig").splitlines()[0])

2026-10-18_156_-_Base_de_Dados.csv 122.3 MB BOM UTF-8: True CRLF: True
2026-10-19_156_-_Base_de_Dados.csv 122.9 MB BOM UTF-8: True CRLF: True
2026-10-20_156_-_Base_de_Dados.csv 123.5 MB BOM UTF-8: True CRLF: True
Tipo;Orgao;DataCriacao;Assunto;Subdivisao;Situacao;Logradouro;Bairro;Regional;DataResposta;Origem;


O arquivo é **UTF-8 com BOM** (e não latin-1), tem quebras de linha CRLF e usa `;` como separador.
Cada linha termina com um `;` extra, e é isso que gera a coluna vazia (`column11` / `Column1`).
A ordem das colunas difere do dicionário: `DataResposta` vem antes de `Origem`.

In [3]:
for d in DIAS:
    t = "d" + d[-2:]
    con.execute(f"""create or replace table {t} as
        select {COLS} from read_csv('{RAW}/{d}_156_-_Base_de_Dados.csv', delim=';', header=true, all_varchar=true)""")
q("select 'd18' snap, count(*) linhas from d18 union all select 'd19', count(*) from d19 union all select 'd20', count(*) from d20")

,snap,linhas
0,d18,620783
1,d19,623695
2,d20,626517


## 2. Nulos e cardinalidade (snapshot de 20/10)

In [4]:
q(" union all ".join(
    f"select '{c}' coluna, count(*) filter (where nullif(trim({c}), '') is null) nulos, count(distinct {c}) distintos from d20"
    for c in COLS.split(",")))

,coluna,nulos,distintos
0,Tipo,0,7
1,Orgao,0,31
2,DataCriacao,0,293
3,Assunto,0,239
4,Subdivisao,0,810
5,Situacao,0,6
6,Logradouro,1,36186
7,Bairro,93,75
8,Regional,93,10
9,DataResposta,70992,293


- `DataResposta` nula significa que a solicitação ainda não foi respondida (cerca de 11%).
- 93 linhas não têm `Bairro` nem `Regional`, embora o dicionário marque `Bairro` como NOT NULL.
- `Logradouro` tem cerca de 36 mil valores distintos. É dado sensível (LGPD) e não deve ser publicado linha a linha.

## 3. Domínios

In [5]:
for c in ["Tipo", "Situacao", "Origem", "Regional"]:
    display(q(f"select {c}, count(*) n from d20 group by 1 order by n desc"))

,Tipo,n
0,Solicitação,548968
1,Reclamação,55245
2,Denúncia,14162
3,Elogio,6888
4,Sugestão,986
5,Recadastro,175
6,Informação,93


,Situacao,n
0,Concluído,544396
1,Aberto,59853
2,Em andamento,11233
3,Respondido,10295
4,Em avaliação,653
5,Falta de dados,87


,Origem,n
0,Telefone,348535
1,App 156,184020
2,Portal,79417
3,Chat,11731
4,App Curitiba,2814


,Regional,n
0,Unidade Regional Matriz,138131
1,Unidade Regional Boa Vista,93314
2,Unidade Regional Portao,73442
3,Unidade Regional Cajuru,69733
4,Unidade Regional Boqueirao,64933
5,Unidade Regional Pinheirinho,51112
6,Unidade Regional Santa Felicidade,44270
7,Unidade Regional CIC,39484
8,Unidade Regional Bairro Novo,32395
9,Unidade Regional Tatuquara,19610


In [6]:
# Há o mesmo órgão escrito de formas diferentes?
q("""select upper(strip_accents(Orgao)) normalizado, count(distinct Orgao) grafias, list(distinct Orgao) variantes
     from d20 group by 1 having count(distinct Orgao) > 1""")

,normalizado,grafias,variantes


In [7]:
q("select count(distinct Orgao) bruto, count(distinct upper(strip_accents(trim(Orgao)))) normalizado from d20")

,bruto,normalizado
0,31,31


## 4. Datas

In [8]:
q("""select
  count(*) filter (where try_strptime(DataCriacao, '%d/%m/%Y') is null) criacao_invalida,
  count(*) filter (where DataResposta is not null and try_strptime(DataResposta, '%d/%m/%Y') is null) resposta_invalida,
  count(*) filter (where strptime(DataResposta, '%d/%m/%Y') < strptime(DataCriacao, '%d/%m/%Y')) resposta_antes_criacao,
  min(strptime(DataCriacao, '%d/%m/%Y')) criacao_min, max(strptime(DataCriacao, '%d/%m/%Y')) criacao_max
from d20""")

,criacao_invalida,resposta_invalida,resposta_antes_criacao,criacao_min,criacao_max
0,0,0,0,2026-01-01,2026-10-20


In [9]:
q("""select strftime(strptime(DataCriacao, '%d/%m/%Y'), '%Y-%m') mes, count(*) solicitacoes,
  median(date_diff('day', strptime(DataCriacao, '%d/%m/%Y'), strptime(DataResposta, '%d/%m/%Y'))) mediana_dias_resposta
from d20 group by 1 order by 1""")

,mes,solicitacoes,mediana_dias_resposta
0,2026-01,67861,10.0
1,2026-02,72998,11.0
2,2026-03,77755,12.0
3,2026-04,63179,10.0
4,2026-05,60190,8.0
5,2026-06,58601,7.0
6,2026-07,61504,7.0
7,2026-08,62039,6.0
8,2026-09,61569,5.0
9,2026-10,40821,1.0


As datas estão em `dd/mm/aaaa`, **sem hora**, e são todas válidas. Nenhuma resposta é anterior à criação.
O snapshot cobre de 01/01 até o próprio dia do arquivo (publicado às 23h30), confirmando que é um **acumulado do ano corrente**.
A mediana de dias até a resposta considera só as solicitações já respondidas, por isso cai nos meses recentes (viés de censura). O dbt precisa tratar isso.
Sem hora, o painel "dia da semana × hora" previsto no plano não é viável e passa a ser "dia da semana × mês".

## 5. Existe chave?

In [10]:
def duplicidade(cols, t="d20"):
    return q(f"""select '{cols[:40]}…' colunas, sum(n) - count(*) linhas_excedentes, count(*) grupos_repetidos, max(n) maior_grupo
               from (select count(*) n from {t} group by {cols}) where n > 1""")

display(duplicidade(COLS))
display(duplicidade(ESTAVEIS))

,colunas,linhas_excedentes,grupos_repetidos,maior_grupo
0,"Tipo,Orgao,DataCriacao,Assunto,Subdivisa…",38882.0,26515,59


,colunas,linhas_excedentes,grupos_repetidos,maior_grupo
0,"Tipo,Orgao,DataCriacao,Assunto,Subdivisa…",50822.0,33307,59


- **Linhas idênticas em todas as colunas:** cerca de 39 mil excedentes, em cerca de 26 mil grupos. Como não há hora nem protocolo, pedidos diferentes ficam indistinguíveis. Não é erro de carga: são solicitações reais. Deduplicar subcontaria a demanda.
- Usando só os campos estáveis (sem `Situacao` e `DataResposta`), há grupos de até 59 linhas iguais.

## 6. Como um snapshot difere do anterior

In [11]:
def diff(a, b, cols):
    return q(f"""select
      (select count(*) from (select {cols} from {a} except all select {cols} from {b})) sairam,
      (select count(*) from (select {cols} from {b} except all select {cols} from {a})) entraram""")

for a, b in [("d18", "d19"), ("d19", "d20")]:
    print(a, "→", b)
    display(diff(a, b, COLS).assign(base="todas as colunas"))
    display(diff(a, b, ESTAVEIS).assign(base="campos estáveis"))

d18 → d19


,sairam,entraram,base
0,3766,6678,todas as colunas


,sairam,entraram,base
0,142,3054,campos estáveis


d19 → d20


,sairam,entraram,base
0,5368,8190,todas as colunas


,sairam,entraram,base
0,275,3097,campos estáveis


In [12]:
# Linhas que "saíram" pelos campos estáveis: edição de órgão, logradouro, assunto etc. (ou exclusão)
q(f"select * from (select {ESTAVEIS} from d19 except all select {ESTAVEIS} from d20) limit 5")

,Tipo,Orgao,DataCriacao,Assunto,Subdivisao,Logradouro,Bairro,Regional,Origem
0,Solicitação,Secretaria Municipal de Obras Públicas,11/10/2026,Fiscalização,Pavimentação definitiva ou alternativa (obra e...,BR-116,JARDIM BOTANICO,Unidade Regional Matriz,App 156
1,Denúncia,Secretaria Municipal do Meio Ambiente,18/10/2026,Poluição,Residual,David de Souza Camargo,PRADO VELHO,Unidade Regional Matriz,Portal
2,Reclamação,SECRETARIA MUNICIPAL DA SAÚDE,09/04/2026,Unidade de Pronto Atendimento 24h - UPA,Outras Ocorrências,RUA ENGENHEIRO BENEDITO MARIO DA SILVA,CAJURU,Unidade Regional Cajuru,Telefone
3,Reclamação,SECRETARIA MUNICIPAL DA SAÚDE,25/03/2026,Unidade de Pronto Atendimento 24h - UPA,Demora no Atendimento,RUA ENGENHEIRO BENEDITO MARIO DA SILVA,CAJURU,Unidade Regional Cajuru,Telefone
4,Reclamação,Secretaria Municipal do Urbanismo,14/09/2026,Passeio,Fiscalização de construção/reconstrução de pas...,Marechal Floriano Peixoto,CENTRO,Unidade Regional Matriz,Portal


In [13]:
# Novas linhas por data de criação: quase todas do dia anterior, mas há inserções retroativas
q(f"""select DataCriacao, count(*) n from (select {ESTAVEIS} from d20 except all select {ESTAVEIS} from d19)
     group by 1 order by n desc limit 10""")

,DataCriacao,n
0,20/10/2026,2814
1,19/10/2026,70
2,16/10/2026,36
3,17/10/2026,33
4,18/10/2026,23
5,15/10/2026,20
6,13/10/2026,13
7,14/10/2026,10
8,07/10/2026,6
9,26/03/2026,4


In [14]:
# Evolução agregada da situação entre snapshots: é o histórico que conseguimos guardar com segurança
q("""select Situacao,
  (select count(*) from d18 x where x.Situacao = s.Situacao) d18,
  (select count(*) from d19 x where x.Situacao = s.Situacao) d19,
  (select count(*) from d20 x where x.Situacao = s.Situacao) d20
from (select distinct Situacao from d20) s order by d20 desc""")

,Situacao,d18,d19,d20
0,Concluído,540822,541848,544396
1,Aberto,59595,59652,59853
2,Em andamento,11648,11382,11233
3,Respondido,8033,9977,10295
4,Em avaliação,604,740,653
5,Falta de dados,81,96,87


In [15]:
# A ordem das linhas não segue a data: o número de ordem não serve como desempate estável
q("select DataCriacao from d20 limit 5")

,DataCriacao
0,06/08/2026
1,06/08/2026
2,07/08/2026
3,07/08/2026
4,07/08/2026


## 7. Conclusões

| Tema | Achado | Consequência no projeto |
| --- | --- | --- |
| Formato | UTF-8 com BOM, CRLF, `;`, coluna vazia no fim | Ler com `utf-8-sig` e descartar a última coluna; o contrato de entrada valida as 11 colunas nomeadas |
| Datas | `dd/mm/aaaa` sem hora, todas válidas | Tipar como `DATE`; painel de hora removido |
| Cobertura | Acumulado do ano, com cerca de 2,8 mil linhas novas por dia | Checagem de volume ≥ snapshot anterior (exceto na virada do ano) |
| Chave | Não existe; há cerca de 39 mil linhas idênticas legítimas | **Não deduplicar.** Ver ADR 0001 |
| Estabilidade | Cerca de 0,05% das linhas mudam campos "estáveis" de um dia para o outro | Rastreio individual (SCD2) seria ruidoso |
| Qualidade | Cada órgão tem grafia única, mas a caixa varia entre órgãos (ex.: `SECRETARIA MUNICIPAL DA SAÚDE`); `Logradouro` também mistura caixa; 93 linhas sem bairro/regional; `Regional` sem acentos | Padronizar a caixa no staging, com seed de nomes de exibição; teste `not_null` de bairro com severidade *warn* |
| LGPD | `Logradouro` tem endereço | Publicar só agregados por bairro |
